# 평가 노트북 (12_eval)

학습된 모델(또는 백본)을 벤치마크에 평가한다. CLI `scripts/eval.sh` 와 **동일한 로직**
(`project.evaluation.evaluate_on_benchmark_suite`)을 노트북에서 호출만 한다.

- 평가 설정: `BENCHMARK/configs/eval/*.yaml`
- 결과: `BENCHMARK/results/<recognizer.name>/`

> 핵심 로직은 `project/` 모듈에. 노트북은 *호출만* (SoT 일원화).

In [ ]:
# 셀 1) 환경 — REPO 루트를 import 경로에 추가
import sys
from pathlib import Path
import yaml

REPO = Path.cwd()
while not (REPO / "project").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("REPO:", REPO)

In [ ]:
# 셀 2) 평가 설정 — 여기만 바꿔서 쓴다
EVAL_CONFIG = REPO / "BENCHMARK/configs/eval/whisper_baseline.yaml"
STAGE = "gold"            # gold | silver  (벤치마크 단계 경로)
BENCH_ROOT = None          # None 이면 /data/ASR/BENCHMARK/<STAGE>. 로컬 샘플은 REPO/"BENCHMARK/data"

# 학습 직후 outputs/<exp> 를 평가할 때만 덮어쓰기 (아니면 None)
MODEL_PATH_OVERRIDE = None   # 예: REPO / "outputs/<exp>"
NAME_OVERRIDE = None         # 예: "<exp>"  (results/<name> 폴더)

cfg = yaml.safe_load(Path(EVAL_CONFIG).read_text())
rec = dict(cfg["recognizer"])
if MODEL_PATH_OVERRIDE:
    rec["model_path"] = str(MODEL_PATH_OVERRIDE)
if NAME_OVERRIDE:
    rec["name"] = NAME_OVERRIDE
print("recognizer:", rec["name"], "| type:", rec["type"], "| model_path:", rec["model_path"])
print("benchmarks:", cfg["benchmarks"], "| stage:", STAGE)

In [ ]:
# 셀 3) 어댑터 분기 → predict_fn (모델 로드)
if rec["type"] == "whisper":
    from project.data.adapters.whisper import build_predict_fn
    predict_fn = build_predict_fn(
        rec["model_path"],
        backbone=rec.get("backbone", rec["model_path"]),
        **rec.get("options", {}),
    )
elif rec["type"] == "sensevoice":
    from project.data.adapters.sensevoice import build_predict_fn
    predict_fn = build_predict_fn(rec["model_path"], **rec.get("options", {}))
else:
    raise ValueError(f"unknown recognizer.type: {rec['type']}")
print("predict_fn ready")

In [ ]:
# 셀 4) 벤치마크 ID → transcript.jsonl 경로 (Fail Fast)
root = Path(BENCH_ROOT) if BENCH_ROOT else Path("/data/ASR/BENCHMARK") / STAGE.upper()
benchmark_paths = {}
for bid in cfg["benchmarks"]:
    p = root / bid / "transcript.jsonl"
    if not p.exists():
        raise FileNotFoundError(
            f"벤치마크 없음: {p}\n  - STAGE(gold/silver) 또는 BENCH_ROOT 를 확인하세요.\n"
            f"  - 로컬 샘플이면 BENCH_ROOT = REPO/'BENCHMARK/data'"
        )
    benchmark_paths[bid] = str(p)
benchmark_paths

In [ ]:
# 셀 5) 평가 실행 → BENCHMARK/results/<name>/
from project.evaluation import evaluate_on_benchmark_suite

out_dir = REPO / "BENCHMARK" / "results" / rec["name"]
results = evaluate_on_benchmark_suite(
    model_name=rec["name"],
    predict_fn=predict_fn,
    benchmark_paths=benchmark_paths,
    out_dir=out_dir,
    batch_size=cfg.get("batch_size", 16),
    save_diff=True,
)

print(f"{'benchmark':35s} {'CER%':>7} {'sCER%':>7} {'n':>6}")
print("-" * 60)
for bid, r in results.items():
    print(f"{bid:35s} {r.cer:7.2f} {r.scer:7.2f} {r.samples:6d}")
print("\n리포트:", out_dir)

In [ ]:
# 셀 6) (선택) 사람 읽는 리포트 + 오답 diff 미리보기
print((out_dir / "evaluation_report.txt").read_text())
print("\n--- diff (앞부분) ---")
diff = out_dir / f"{rec['name']}_diff.txt"
if diff.exists():
    print("\n".join(diff.read_text().splitlines()[:25]))